# G-Retriever + Linear Module — K-Fold Cross-Validation Pipeline

- **Setup & data**: locates the project root / `DATA` folder, loads the config for a single category (`artesania`), and loads the MCQ questions filtered by the article subset.
- **Artifacts**: builds/loads the graph artifacts and the RAG index, then loads the LLM + tokenizer used across the whole pipeline.
- **PCST cache**: reloads a precomputed PCST cache (one retrieved subgraph + textual description per question) so retrieval isn't recomputed.
- **K-Fold CV**: runs a stratified (by category) k-fold cross-validation with crash-safe incremental saving / resuming, then aggregates results per ablation and runs paired Wilcoxon tests across folds.

In [ ]:
from pathlib import Path
ROOT = Path().resolve()
while not (ROOT / "DATA").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
DATA = ROOT / "DATA"
print("CLEAN root:", ROOT)

In [ ]:
import os
import sys
sys.path.insert(0, "..")  # to import src/

os.environ["PYTORCH_CUDA_ALLOC_CONF"]="expandable_segments:True"

import torch
from sentence_transformers import SentenceTransformer

from src.config import Config
from src.data_loading import load_questions, load_article_ids, filter_questions_by_articles, sample_df
from src.graph_utils import ensure_graph_artifacts

from src.retrieval_rag import ensure_rag_index
from src.llm import load_llm
from src.evaluation import run_all_modes
from src.inspection import inspect_examples
from src.io_utils import save_results
from src.filter_components import filter_graph_json_to_giant_component

In [ ]:
print("CUDA available:", torch.cuda.is_available())
print("CUDA device count:", torch.cuda.device_count())
if torch.cuda.is_available():
    print("CUDA device0:", torch.cuda.get_device_name(0))

In [ ]:
CATEGORY = "artesania"

cfg = Config(
    CATEGORY=CATEGORY,
    graph_dir=str(DATA / "SUBSETS_ES" / "GRAPHS" / CATEGORY),  # Correction ici
    n_samples=None,
    modes_to_run=["zero_shot", "graph_rag_pcst", "rag", "kaping"],
    force_rebuild_graph_artifacts=False,
    force_rebuild_rag_index=False,
    device="cuda:0"
)
print(cfg)

In [ ]:
df_questions = load_questions(cfg.questions_csv)

# Keep only the questions whose source article belongs to the selected subset
if cfg.filter_by_articles:
    article_ids = load_article_ids(cfg.articles_csv)
    df_questions = filter_questions_by_articles(df_questions, article_ids)

# ⚠️ Edit this cell if you want to change the evaluated dataframe
df_eval = sample_df(df_questions, cfg.n_samples, cfg.random_seed)
print(f"📊 df_eval : {len(df_eval)} questions")

## LOADING EMBEDDERS 

In [ ]:
# Sentence embedder used to embed graph nodes/edges
graph_embedder = SentenceTransformer(cfg.graph_embedder_id, device=cfg.device, trust_remote_code=True)
nodes_df, edges_df, node2id, graph = ensure_graph_artifacts(cfg, graph_embedder)

rag_embedder = SentenceTransformer(cfg.rag_embedder_id, device=cfg.device, trust_remote_code=True)
rag_index, rag_chunks_df = ensure_rag_index(cfg)  # builds the index only if missing / forced


## LOADING LLM

In [ ]:
llm, tokenizer = load_llm(cfg.llm_id, device=cfg.device)

## CREATE PCST cache

In [ ]:
# from src.gnn_utils.pcst_cache import build_pcst_cache
# from src.gnn_utils.gnn_config import GNNConfig

# pcst_cache_dir= cfg.graph_dir + "/pcst_cache_for_gnn"
# os.makedirs(pcst_cache_dir, exist_ok=True)

# pcst_cache_path = pcst_cache_dir + "/pcst_cache.pkl"
# gcfg = GNNConfig(category="global",device="cuda:1",batch_size=1,pcst_cache_path=pcst_cache_path)

# # 1) PCST cache (computed once)
# pcst_cache = build_pcst_cache(
#     df_questions, cfg, graph, nodes_df, edges_df,
#     graph_embedder, gcfg.pcst_cache_path,
# )

In [ ]:
# LOAD THE PCST CACHE PICKLE:
import pickle
pcst_cache_path = cfg.graph_dir + "/pcst_cache_for_gnn/pcst_cache.pkl"
with open(pcst_cache_path, "rb") as f:
    pcst_cache = pickle.load(f)   # one retrieved subgraph + description per question

## K-Fold Cross-Validation

Same stratified k-fold cross-validation setup as the main GNN pipeline, but here the GNN encoder inside `GRetriever` is swapped for a simple **`LinearModel`** (no message passing) whenever `use_graph_token=True`.

- Folds are generated **once** (stratified by `category`, `K_FOLDS = 5`) and reused for every ablation.
- Training is **crash-safe**: each finished run is appended to `kfold_details.csv` immediately, and failed runs are logged then automatically retried on re-run.
- Results are saved under `RESULTS_LINEAR` (separate from the GNN results) and aggregated per ablation, with paired Wilcoxon tests on folds completed by all ablations.

In [ ]:
import copy
import numpy as np
import pandas as pd
import torch
from pathlib import Path
from datetime import datetime
from sklearn.model_selection import StratifiedKFold, train_test_split
from scipy import stats
import traceback

from src.gnn_utils.gnn_config import GNNConfig
from src.gnn_utils.g_retriever_model import GRetriever
from src.gnn_utils.train import train_g_retriever
from src.gnn_utils.dataset import GRetrieverMCQDataset
from src.gnn_utils.linear_model import LinearModel  # simple non-graph encoder used as ablation

K_FOLDS = 5
SEED    = 1

RESULTS_LINEAR_ROOT = str(DATA / "RESULTS_LINEAR")  # dedicated results folder for this ablation

dataset = GRetrieverMCQDataset(df_questions, pcst_cache, graph)
n_total = len(dataset)

# Stratify by category (fallback on correct_letter if not available)
if "category" in df_questions.columns:
    strat_labels = df_questions["category"].to_numpy()
    print(f"Stratifying by 'category' : {pd.Series(strat_labels).value_counts().to_dict()}")
else:
    strat_labels = df_questions["correct_letter"].to_numpy()
    print("Stratifying by 'correct_letter' (fallback)")

def make_kfold_splits(n, labels, k, val_ratio, seed):
    """Return a list of k (train_idx, val_idx, test_idx) tuples, stratified.
    The val set is carved out of each fold's train set (also stratified)."""
    skf = StratifiedKFold(n_splits=k, shuffle=True, random_state=seed)
    all_idx = np.arange(n)
    folds = []
    for trainval_idx, test_idx in skf.split(all_idx, labels):
        rel_val = val_ratio / (1.0 - 1.0 / k)
        train_idx, val_idx = train_test_split(
            trainval_idx,
            test_size=rel_val,
            stratify=labels[trainval_idx],
            random_state=seed,
        )
        folds.append((
            np.sort(train_idx).tolist(),
            np.sort(val_idx).tolist(),
            np.sort(test_idx).tolist(),
        ))
    return folds

_ref_cfg = GNNConfig(category="global", batch_size=1)
kfold_splits = make_kfold_splits(
    n_total, strat_labels, k=K_FOLDS, val_ratio=_ref_cfg.val_ratio, seed=SEED
)

print(f"\nK-Fold ({K_FOLDS} folds) — sizes [train/val/test] :")
for i, (tr, va, te) in enumerate(kfold_splits):
    print(f"  fold {i}: {len(tr)} / {len(va)} / {len(te)}")

# Base config — hyperparameters frozen, only the encoder changes across ablations
base_cfg = GNNConfig(
    train_gen_mode="gen_letter",
    use_graph_token=True,
    use_text_graph=True,
    category="global",
    lr=1e-4,
    results_root=RESULTS_LINEAR_ROOT,
    device="cuda:0",
    batch_size=2
)

ablations = {
    # "text_only":  dict(use_graph_token=False, use_text_graph=True),
    "both":       dict(use_graph_token=True,  use_text_graph=True),
    # "graph_only": dict(use_graph_token=True,  use_text_graph=False),
}

llm_slang = llm.name_or_path.split("/")[-1].replace("-", "_")
out_dir = Path(RESULTS_LINEAR_ROOT) / llm_slang / "gen_letter" / base_cfg.category / "kfold_cv"
out_dir.mkdir(parents=True, exist_ok=True)

progress_csv = out_dir / "kfold_details.csv"   # updated after EVERY run
splits_path  = out_dir / "kfold_splits.npy"

# save the folds right away (reproducibility + resuming)
np.save(splits_path, np.array(kfold_splits, dtype=object), allow_pickle=True)

# --- RESUME: reload what has already been done ---
if progress_csv.exists():
    cv_df = pd.read_csv(progress_csv)
    if "status" in cv_df.columns:
        done_mask = cv_df["status"] == "ok"
    else:
        done_mask = pd.Series(False, index=cv_df.index)
    done = set(zip(cv_df.loc[done_mask, "fold"], cv_df.loc[done_mask, "ablation"]))
    to_retry = set(zip(cv_df.loc[~done_mask, "fold"], cv_df.loc[~done_mask, "ablation"]))
    to_retry -= done  # runs to redo: present in CSV but status != "ok"
    print(f"♻️  Resuming : {len(done)} 'ok' run(s) -> {sorted(done)}")
    if to_retry:
        print(f"🔁 To re-run (status != 'ok') : {sorted(to_retry)}")
else:
    cv_df = pd.DataFrame()
    done = set()

def append_row(row: dict):
    """Add/replace a (fold, ablation) row and rewrite the CSV IMMEDIATELY (crash-safe)."""
    global cv_df
    if not cv_df.empty and {"fold", "ablation"}.issubset(cv_df.columns):
        cv_df = cv_df[~(
            (cv_df["fold"] == row["fold"]) &
            (cv_df["ablation"] == row["ablation"])
        )].copy()
    cv_df = pd.concat([cv_df, pd.DataFrame([row])], ignore_index=True)
    # atomic write: tmp then rename (avoids a corrupted CSV if we crash mid-write)
    tmp = progress_csv.with_suffix(".csv.tmp")
    cv_df.to_csv(tmp, index=False)
    tmp.replace(progress_csv)

failures = []

for fold_i, split in enumerate(kfold_splits):
    print(f"\n{'#'*70}\n# FOLD {fold_i+1}/{K_FOLDS}\n{'#'*70}")

    for name, flags in ablations.items():
        # --- SKIP if already done (resume) ---
        if (fold_i, name) in done:
            print(f"⏭  skip fold={fold_i} ablation={name} (already done)")
            continue

        print(f"\n{'='*60}\n>>> fold={fold_i}  ablation={name}  {flags}\n{'='*60}")

        try:
            cfg = copy.deepcopy(base_cfg)
            cfg.use_graph_token = flags["use_graph_token"]
            cfg.use_text_graph  = flags["use_text_graph"]
            cfg.ablation_name   = f"cv_fold{fold_i}_{name}"
            cfg.ckpt_path = base_cfg.ckpt_path.replace(".pt", f"_fold{fold_i}_{name}.pt")

            model = GRetriever(llm, tokenizer, cfg)
            model.llm.gradient_checkpointing_enable()
            model.llm.config.use_cache = False

            # Swap the default GNN encoder for a plain LinearModel (no graph structure)
            if flags["use_graph_token"]:
                model.gnn = LinearModel(
                    in_dim=cfg.in_dim, hidden_dim=cfg.hidden_dim,
                    out_dim=cfg.gnn_out_dim, edge_dim=cfg.edge_dim,
                    num_layers=cfg.num_layers, dropout=cfg.dropout,
                )

            res = train_g_retriever(
                model, df_questions, pcst_cache, graph, cfg,
                splits=split,
            )

            append_row({
                "fold": fold_i,
                "ablation": name,
                "val_acc":   res["best_val_acc"],
                "test_acc":  res["test_acc"],
                "best_epoch": res.get("best_epoch", 0),
                "run_dir":   res.get("run_dir", None),
                "status":    "ok",
                "timestamp": datetime.now().isoformat(timespec="seconds"),
            })
            print(f"💾 saved -> {progress_csv.name} ({len(cv_df)} run(s) in total)")

            del model
            torch.cuda.empty_cache()

        except Exception as e:
            # --- a run crashed: log it, free memory, and KEEP GOING ---
            tb = traceback.format_exc()
            print(f"❌ FAILED fold={fold_i} ablation={name} : {e}")
            print(tb)
            failures.append((fold_i, name, str(e)))

            append_row({
                "fold": fold_i,
                "ablation": name,
                "val_acc": None, "test_acc": None, "best_epoch": None,
                "run_dir": None, "status": f"FAILED: {e}",
                "timestamp": datetime.now().isoformat(timespec="seconds"),
            })

            # detailed log in a separate file
            with open(out_dir / "failures.log", "a") as f:
                f.write(f"\n{'='*60}\nfold={fold_i} ablation={name} "
                        f"@ {datetime.now().isoformat()}\n{tb}\n")

            try:
                del model
            except NameError:
                pass
            torch.cuda.empty_cache()

# --- FINAL aggregation (failed runs are ignored) ---
ok_df = cv_df[cv_df["status"] == "ok"].copy()
ok_df["test_acc"] = pd.to_numeric(ok_df["test_acc"])

if failures:
    print(f"\n⚠️  {len(failures)} failed run(s) : {failures}")
    print("   -> just re-run this cell: they will be retried, the 'ok' ones will be skipped.")

summary = (
    ok_df.groupby("ablation")["test_acc"]
    .agg(test_acc_mean="mean", test_acc_std="std",
         test_acc_min="min", test_acc_max="max", n="count")
    .sort_values("test_acc_mean", ascending=False)
)
print(f"\n{'='*70}\n📊 K-FOLD SUMMARY ({K_FOLDS} folds)\n{'='*70}")
print(summary.to_string())
summary.to_csv(out_dir / "kfold_summary.csv")

# pivot only on COMPLETE folds (all ablations ok)
complete_folds = (
    ok_df.groupby("fold")["ablation"].nunique()
    .loc[lambda s: s == len(ablations)].index
)
if len(complete_folds) >= 2:
    print("\nPer-fold detail (complete folds, test_acc) :")
    pivot = (ok_df[ok_df["fold"].isin(complete_folds)]
             .pivot(index="fold", columns="ablation", values="test_acc"))
    print(pivot.to_string())

    # paired Wilcoxon tests on complete folds
    print(f"\n🔬 Paired tests (Wilcoxon, {len(complete_folds)} complete folds)")
    abl_names = list(pivot.columns)
    for i in range(len(abl_names)):
        for j in range(i + 1, len(abl_names)):
            a, b = abl_names[i], abl_names[j]
            try:
                w, p = stats.wilcoxon(pivot[a], pivot[b])
                print(f"  {a} vs {b}: W={w:.2f} p={p:.4f} (Δmean={pivot[a].mean()-pivot[b].mean():+.4f})")
            except ValueError as e:
                print(f"  {a} vs {b}: N/A ({e})")
else:
    print("\n⚠️  Not enough complete folds for the paired test.")

print(f"\n✅ Everything saved in : {out_dir}")